In [13]:
import pandas as pd
# test = pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/lineups/2026-02-24/lineups_013149.parquet')
pd.set_option('display.max_columns', None)

In [ ]:
pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/schedules/games_2000.parquet')

In [ ]:
# =============================================
# NOTEBOOK SETUP — Run this cell first
# =============================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # goes up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print("✅ Project root:", PROJECT_ROOT)
print("✅ Connecting to DuckDB (read-only):", DB_PATH)

# read_only=True = no more lock errors for analysis
con = duckdb.connect(str(DB_PATH), read_only=True)

print("✅ Connected successfully in read-only mode!")
print("Total games in silver_lineups:", 
      con.sql("SELECT COUNT(*) as total FROM silver_lineups").df().iloc[0]['total'])

Day By Day Stat Download Testing

In [9]:
# ────────────────────────────────────────────────
# Cell 1: Basic imports + one-day batting totals
# ────────────────────────────────────────────────

import pybaseball as pb
import pandas as pd

# Example: get all MLB batting stats that occurred on July 4, 2024
start_date = "2025-02-20"
# end_date   = "2024-07-04"   # same date = single day
end_date = start_date

daily_batting = pb.batting_stats_range(start_date, end_date)

print(f"Shape: {daily_batting.shape}")
display(daily_batting.head(8))          # first few rows (player-level)
print("\nColumns:", daily_batting.columns.tolist())

# Quick team totals for that day
team_totals = daily_batting.groupby('Tm').agg({
    'G': 'sum', 'PA': 'sum', 'AB': 'sum', 'R': 'sum', 'H': 'sum',
    'HR': 'sum', 'RBI': 'sum', 'BB': 'sum', 'SO': 'sum'
}).sort_values('R', ascending=False)

print("\nTeam Batting Totals on", start_date)
display(team_totals)

IndexError: list index out of range

In [ ]:
# ────────────────────────────────────────────────
# Cell 2: Same thing but for pitching stats on a date
# ────────────────────────────────────────────────

daily_pitching = pb.pitching_stats_range(start_date, end_date)

team_pitch_totals = daily_pitching.groupby('Tm').agg({
    'G': 'sum', 'IP': 'sum', 'H': 'sum', 'R': 'sum', 'ER': 'sum',
    'HR': 'sum', 'BB': 'sum', 'SO': 'sum', 'ERA': 'mean'   # or weighted later
}).sort_values('R', ascending=True)   # lower runs allowed = better

print("\nTeam Pitching Totals Allowed on", start_date)
display(team_pitch_totals)

In [ ]:
# ────────────────────────────────────────────────
# Cell 3: One-liner function you can reuse
# ────────────────────────────────────────────────

def get_daily_team_totals(date_str: str, stat_type='batting'):
    """
    date_str: 'YYYY-MM-DD'
    stat_type: 'batting' or 'pitching'
    """
    start = end = date_str
    
    if stat_type == 'batting':
        df = pb.batting_stats_range(start, end)
        key_stats = ['G','PA','AB','R','H','2B','3B','HR','RBI','BB','SO']
    elif stat_type == 'pitching':
        df = pb.pitching_stats_range(start, end)
        key_stats = ['G','IP','H','R','ER','HR','BB','SO']
    else:
        raise ValueError("stat_type must be 'batting' or 'pitching'")
    
    if df.empty:
        print(f"No data found for {date_str}")
        return pd.DataFrame()
    
    team_df = df.groupby('Tm', as_index=False)[key_stats].sum()
    team_df = team_df.sort_values('R', ascending=(stat_type=='pitching'))
    
    print(f"{stat_type.title()} team totals for {date_str} — {len(team_df)} teams")
    return team_df

# Usage examples
display(get_daily_team_totals("2025-08-15", "batting"))
# display(get_daily_team_totals("2023-10-01", "pitching"))

In [59]:
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/raw/schedules/games_2026.parquet')
batpath = '/Users/patrickmaloney/Documents/mlb-betting/data/raw/player_logs/game_by_game/pitching_game_logs_2025.parquet'
batting = pd.read_parquet(batpath)
lineup = pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/bronze/lineups/2026-03-08/lineups_201831.parquet')
lahman = pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/reference/lahman_files/people.parquet')

In [61]:
lahman.head(4).to_clipboard()

In [25]:
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/reference/lahman_files/historical_teams_data.parquet')[['yearID', 'lgID', 'teamID', 'divID', 'teamIDBR']].drop_duplicates()

,yearID,lgID,teamID,divID,teamIDBR
0,1871,NaN,BS1,NaN,BOS
1,1871,NaN,CH1,NaN,CHI
2,1871,NaN,CL1,NaN,CLE
3,1871,NaN,FW1,NaN,KEK
4,1871,NaN,NY2,NaN,NYU
...,...,...,...,...,...
3609,2025,NL,SLN,C,STL
3610,2025,AL,TBA,E,TBR
3611,2025,AL,TEX,W,TEX
3612,2025,AL,TOR,E,TOR
